In [ ]:
import torch
import numpy as np
from pathlib import Path
from torch.optim import Adam
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
from rt_whisper.models import BoundaryWordFilter

In [ ]:
EXPONENT = 2

In [ ]:
head_model = BoundaryWordFilter()
tail_model = BoundaryWordFilter()

In [ ]:
@torch.no_grad()
def flatten_params() -> torch.Tensor:
    return torch.cat(
        [p.data.view(-1) for p in head_model.parameters()] +
        [p.data.view(-1) for p in tail_model.parameters()]
    )

@torch.no_grad()
def assign_flat_params(flat: np.ndarray) -> None:
    off = 0
    for m in [head_model, tail_model]:
        for p in m.parameters():
            n = p.numel()
            p.data.copy_(torch.from_numpy(flat[off:off+n]).view_as(p))
            off += n

In [ ]:
def generate_data(size:int, boundary:int = 16000, dur:int = None):
    for _ in range(size):
        start, end = sorted(np.random.randint(0, boundary, 2))
        if dur is not None:
            if start + dur <= boundary:
                end = start + dur
            else:
                start = end - dur

        start = start // 160 * 160
        end = end // 160 * 160
        mid = (start + end) / 2
        dur = end - start
        weight = (mid/boundary) ** EXPONENT

        start /= boundary
        end /= boundary
        mid /= boundary
        dur /= boundary

        yield (start, end, mid, dur), weight

In [ ]:
def train_mae(
    model: nn.Module,
    X: torch.Tensor,
    Y: torch.Tensor,
    epochs: int = 10000,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    device: str | torch.device = "cpu",
    log_interval: int = 100,
    patience: int = 500,
    improvement_tol: float = 0.0,
    batch_size: int = 256,
    num_workers: int = 0,
    pin_memory: bool = False,
):
    model.to(device)
    opt = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    mae = nn.L1Loss(reduction="none")

    ds = TensorDataset(X, Y)
    dl = DataLoader(
        ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory
    )

    best_loss = float("inf")
    best_sd = None
    no_improve = 0

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n_seen = 0

        for xb, yb in dl:
            xb = xb.to(device, non_blocking=pin_memory)
            yb = yb.to(device, non_blocking=pin_memory)

            pred = model(xb)
            loss = mae(pred, yb).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = xb.size(0)
            running_loss += loss.detach().item() * bs
            n_seen += bs

        epoch_loss = running_loss / max(1, n_seen)

        improved = (best_loss - epoch_loss) > improvement_tol
        if improved:
            best_loss = epoch_loss
            best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if (ep % log_interval == 0) or (ep == 1) or (ep == epochs) or improved:
            print(f"[{ep:04d}] loss={epoch_loss:.6f}  best={best_loss:.6f}  no_improve={no_improve}")

        if no_improve >= patience:
            print(f"Early stop at epoch {ep} (no improvement for {patience} epochs).")
            break

    model.eval()
    if best_sd is not None:
        model.load_state_dict(best_sd)
    return model

In [ ]:
def plot_mid_vs_pred_and_target(
    model: nn.Module,
    X: torch.Tensor,
    Y: torch.Tensor,
    device: str | torch.device = "cpu"
):
    model.to(device)
    X = X.to(device)
    Y = Y.to(device)

    with torch.no_grad():
        # mid만 추출 (X의 세 번째 컬럼)
        mids = X[:, 2].cpu().numpy()

        # 모델 예측
        preds = model(X).cpu().numpy().flatten()
        Y = Y.cpu().numpy().flatten()

    # 산점도 그리기
    plt.figure(figsize=(8,5))
    plt.scatter(mids, Y, alpha=0.5, label="Target (Y)", color="blue")
    plt.scatter(mids, preds, alpha=0.5, label="Model Output", color="red")
    plt.xlabel("mid")
    plt.ylabel("value")
    plt.title("Mid vs Model Prediction & Target")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

In [ ]:
dataset = set()
while True:
    for d in generate_data(size=2000):
        dataset.add(d)
    if len(dataset) >= 2000:
        dataset = list(dataset)
        dataset = dataset[:2000]
        break

In [ ]:
X, Y = zip(*dataset)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
x_train = torch.tensor(x_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [ ]:
head_model = train_mae(head_model, x_train, y_train, device="cuda")

In [ ]:
tail_model = train_mae(tail_model, x_train, y_train, device="cuda")

In [ ]:
plot_mid_vs_pred_and_target(head_model, x_test, y_test, device="cuda")
plot_mid_vs_pred_and_target(tail_model, x_test, y_test, device="cuda")

In [ ]:
torch.save(head_model, "/workspaces/dev/test/optimize/all/model_weight/head_model-pt-(m).pth")
torch.save(tail_model, "/workspaces/dev/test/optimize/all/model_weight/tail_model-pt-(m).pth")

In [ ]:
flatten_params().cpu().numpy()